# 06 — Statistics for the completed GPT-5.5 factorial

Rebuilds every table now that GPT-5.5 covers all ten prompt conditions
(3600 rows). Conventions are inherited from the published tables so the three
model generations remain directly comparable:

* **McNemar** — exact two-sided binomial test on discordant pairs.
* **Holm** — applied *within* each model over its own 45 pairwise comparisons.
* **Odds ratio (paired)** — ratio of discordant counts, b / c.
* **chi2_cc** — continuity-corrected McNemar statistic, (|b-c|-1)^2 / (b+c).

Purely local; no API calls, safe to re-run.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')
WORK = ROOT / 'revision_2026'
OUT  = WORK / 'tables_r3'
OUT.mkdir(exist_ok=True)

CONDS = [f'{v}_{h}' for v in 'ABCDE' for h in ('clean', 'hinted')]
GPT55 = 'GPT-5.5'

f55 = pd.read_csv(WORK / 'frontier_gpt55.csv')
f55 = f55[f55.run_id == 1].copy()
f55['correct'] = f55.prediction == f55.true_label

assert len(f55) == 3600, f'expected 3600 rows, found {len(f55)}'
assert set(f55.condition) == set(CONDS), 'condition set incomplete'
print('GPT-5.5:', len(f55), 'rows |', f55.condition.nunique(), 'conditions')

GPT-5.5: 3600 rows | 10 conditions


## TABLE I — full metric set

Ten conditions for GPT-5.5, appended to the published legacy rows. Wilson
intervals accompany recall and specificity so the reader can see the precision
behind each point estimate.

In [4]:
def metrics(g):
    tp = int(((g.true_label=='Vulnerable') & (g.prediction=='Vulnerable')).sum())
    fn = int(((g.true_label=='Vulnerable') & (g.prediction=='Safe')).sum())
    tn = int(((g.true_label=='Safe')       & (g.prediction=='Safe')).sum())
    fp = int(((g.true_label=='Safe')       & (g.prediction=='Vulnerable')).sum())
    rec  = tp/(tp+fn)
    spec = tn/(tn+fp)
    prec = tp/(tp+fp) if tp+fp else np.nan
    den  = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))) or np.nan
    rl, rh = proportion_confint(tp, tp+fn, method='wilson')
    sl, sh = proportion_confint(tn, tn+fp, method='wilson')
    return pd.Series({
        'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp,
        'Recall': round(rec,3), 'Recall_lo': round(rl,3), 'Recall_hi': round(rh,3),
        'Specificity': round(spec,3), 'Spec_lo': round(sl,3), 'Spec_hi': round(sh,3),
        'Precision': round(prec,3),
        'F1':     round(2*prec*rec/(prec+rec),3) if prec+rec else np.nan,
        'BalAcc': round((rec+spec)/2, 3),
        'MCC':    round((tp*tn - fp*fn)/den, 3),
    })

t1_55 = (f55.groupby('condition').apply(metrics, include_groups=False)
            .reindex(CONDS).reset_index())
t1_55.insert(0, 'Model', GPT55)
t1_55 = t1_55.rename(columns={'condition': 'Condition'})

print(t1_55.drop(columns=['Recall_lo','Recall_hi','Spec_lo','Spec_hi']).to_string(index=False))
t1_55.to_csv(OUT / 'TABLE1_gpt55_10conditions.csv', index=False)

  Model Condition    TP    FN    TN   FP  Recall  Specificity  Precision    F1  BalAcc   MCC
GPT-5.5   A_clean 162.0  18.0 126.0 54.0   0.900        0.700      0.750 0.818   0.800 0.612
GPT-5.5  A_hinted 104.0  76.0 163.0 17.0   0.578        0.906      0.860 0.691   0.742 0.512
GPT-5.5   B_clean 156.0  24.0 127.0 53.0   0.867        0.706      0.746 0.802   0.786 0.580
GPT-5.5  B_hinted  98.0  82.0 163.0 17.0   0.544        0.906      0.852 0.664   0.725 0.483
GPT-5.5   C_clean 147.0  33.0 132.0 48.0   0.817        0.733      0.754 0.784   0.775 0.552
GPT-5.5  C_hinted 130.0  50.0 155.0 25.0   0.722        0.861      0.839 0.776   0.792 0.589
GPT-5.5   D_clean 140.0  40.0 132.0 48.0   0.778        0.733      0.745 0.761   0.756 0.512
GPT-5.5  D_hinted  79.0 101.0 166.0 14.0   0.439        0.922      0.849 0.579   0.681 0.412
GPT-5.5   E_clean 144.0  36.0 133.0 47.0   0.800        0.739      0.754 0.776   0.769 0.540
GPT-5.5  E_hinted 113.0  67.0 161.0 19.0   0.628        0.894      0.8

## TABLE II — 45 pairwise McNemar tests on GPT-5.5

Same procedure as the legacy models: exact binomial test on discordant pairs,
Holm correction across this model's own 45 comparisons.

In [5]:
wide = f55.pivot_table(index='sample_id', columns='condition',
                       values='correct', aggfunc='first')
wide = wide[CONDS]
assert wide.notna().all().all(), 'missing cells in the condition x sample matrix'

rows = []
for i, ca in enumerate(CONDS):
    for cb in CONDS[i+1:]:
        a_only = int((wide[ca] & ~wide[cb]).sum())   # b
        b_only = int((~wide[ca] & wide[cb]).sum())   # c
        n_disc = a_only + b_only
        p = (stats.binomtest(a_only, n_disc, 0.5).pvalue if n_disc else 1.0)
        chi2 = ((abs(a_only-b_only)-1)**2 / n_disc) if n_disc else np.nan
        rows.append({'Model': GPT55, 'CondA': ca, 'CondB': cb,
                     'A_only': a_only, 'B_only': b_only,
                     'chi2_cc': round(chi2,3) if n_disc else np.nan,
                     'OR': round(a_only/b_only,2) if b_only else np.inf,
                     'p_exact': p})

t2_55 = pd.DataFrame(rows)
_, p_holm, _, _ = multipletests(t2_55.p_exact, method='holm')
t2_55['p_holm']   = np.minimum(p_holm, 1.0)
t2_55['sig_holm'] = t2_55.p_holm < 0.05

print(f'significant after Holm: {int(t2_55.sig_holm.sum())} of 45\n')
print(t2_55[t2_55.sig_holm]
      .sort_values('p_holm')[['CondA','CondB','A_only','B_only','OR','p_exact','p_holm']]
      .to_string(index=False))
t2_55.to_csv(OUT / 'TABLE2_gpt55_pairwise_mcnemar.csv', index=False)

significant after Holm: 6 of 45

   CondA    CondB  A_only  B_only   OR      p_exact   p_holm
C_hinted D_hinted      53      13 4.08 7.237175e-07 0.000033
D_hinted E_hinted      10      39 0.26 3.845912e-05 0.001692
A_hinted D_hinted      27       5 5.40 1.130742e-04 0.004862
 A_clean D_hinted      83      40 2.08 1.319598e-04 0.005542
 B_clean D_hinted      77      39 1.97 5.340946e-04 0.021898
 C_clean D_hinted      70      36 1.94 1.240240e-03 0.049610


### Merge with the published legacy comparisons

Holm was applied per model, so the legacy rows are unaffected by adding a third
model — they can be concatenated unchanged.

In [7]:
import subprocess
print(subprocess.run(['find', str(ROOT), '-name', 'TABLE*.csv'],
                     capture_output=True, text=True).stdout)
print(subprocess.run(['find', str(ROOT), '-name', 'stability*.csv'],
                     capture_output=True, text=True).stdout)

/content/drive/MyDrive/LLM_Security_Paper/revision_2026/tables_r3/TABLE1_gpt55_10conditions.csv
/content/drive/MyDrive/LLM_Security_Paper/revision_2026/tables_r3/TABLE2_gpt55_pairwise_mcnemar.csv

/content/drive/MyDrive/LLM_Security_Paper/revision_2026/stability_merged_3runs.csv
/content/drive/MyDrive/LLM_Security_Paper/revision_2026/stability_gpt55.csv



In [8]:
import urllib.request
BASE = 'https://raw.githubusercontent.com/sam-kao-TW/llm-vuln-detection-ablation/main/data/'
for f in ['TABLE2_pairwise_mcnemar_floored.csv',
          'TABLE4_gpt55_by_sanitizer_mechanism.csv',
          'TABLE5_capability_spectrum.csv']:
    urllib.request.urlretrieve(BASE + f, WORK / f)
    print('downloaded', f)

downloaded TABLE2_pairwise_mcnemar_floored.csv
downloaded TABLE4_gpt55_by_sanitizer_mechanism.csv
downloaded TABLE5_capability_spectrum.csv


In [9]:
legacy2 = pd.read_csv(WORK / 'TABLE2_pairwise_mcnemar_floored.csv')
print('legacy rows:', len(legacy2), '| significant:', int(legacy2.sig_holm.sum()))

t2_all = pd.concat([legacy2, t2_55[legacy2.columns.intersection(t2_55.columns)]],
                   ignore_index=True)
t2_all.to_csv(OUT / 'TABLE2_all_models_pairwise.csv', index=False)
print('combined rows:', len(t2_all), '(expect 135)')
print(t2_all.groupby('Model').sig_holm.agg(['size','sum']))

legacy rows: 90 | significant: 0
combined rows: 135 (expect 135)
                    size  sum
Model                        
GPT-3.5-turbo-0125    45    0
GPT-4o-mini           45    0
GPT-5.5               45    6


## TABLE III — component attribution

The published table contrasted Baseline against the Full Framework only. With
the factorial closed, each component can be isolated against the same baseline,
and category naming can be tested within every structural variant.

In [10]:
def paired(ca, cb, mask=None):
    w = wide if mask is None else wide[mask]
    b = int((w[ca] & ~w[cb]).sum())
    c = int((~w[ca] & w[cb]).sum())
    n = b + c
    return {'n': len(w),
            f'{ca}_correct': int(w[ca].sum()), f'{cb}_correct': int(w[cb].sum()),
            'A_only': b, 'B_only': c,
            'chi2_cc': round(((abs(b-c)-1)**2)/n, 3) if n else np.nan,
            'p_exact': stats.binomtest(b, n, 0.5).pvalue if n else 1.0,
            'OR': round(b/c, 2) if c else np.inf}

vuln = f55[f55.run_id==1].drop_duplicates('sample_id').set_index('sample_id').true_label
is_v = wide.index.map(vuln) == 'Vulnerable'

contrasts  = [('A_clean', v) for v in ['B_clean','C_clean','D_clean','E_clean']]
contrasts += [(f'{v}_clean', f'{v}_hinted') for v in 'ABCDE']

out = []
for ca, cb in contrasts:
    for label, m in [('Overall', None), ('Vulnerable', is_v), ('Safe', ~is_v)]:
        out.append({'Contrast': f'{ca} vs {cb}', 'Stratum': label,
                    **paired(ca, cb, m)})
t3 = pd.DataFrame(out)
t3['p_holm'] = np.minimum(multipletests(t3.p_exact, method='holm')[1], 1.0)
t3['sig_holm'] = t3.p_holm < 0.05

pd.set_option('display.width', 200)
print(t3[['Contrast','Stratum','A_only','B_only','OR','p_exact','p_holm','sig_holm']]
      .to_string(index=False))
t3.to_csv(OUT / 'TABLE3_component_attribution.csv', index=False)

           Contrast    Stratum  A_only  B_only    OR      p_exact       p_holm  sig_holm
 A_clean vs B_clean    Overall      11       6  1.83 3.323059e-01 1.000000e+00     False
 A_clean vs B_clean Vulnerable       9       3  3.00 1.459961e-01 8.759766e-01     False
 A_clean vs B_clean       Safe       2       3  0.67 1.000000e+00 1.000000e+00     False
 A_clean vs C_clean    Overall      20      11  1.82 1.496128e-01 8.759766e-01     False
 A_clean vs C_clean Vulnerable      18       3  6.00 1.489639e-03 2.234459e-02      True
 A_clean vs C_clean       Safe       2       8  0.25 1.093750e-01 7.656250e-01     False
 A_clean vs D_clean    Overall      25       9  2.78 9.041185e-03 1.265766e-01     False
 A_clean vs D_clean Vulnerable      25       3  8.33 2.744049e-05 4.939288e-04      True
 A_clean vs D_clean       Safe       0       6  0.00 3.125000e-02 3.750000e-01     False
 A_clean vs E_clean    Overall      23      12  1.92 8.953108e-02 7.162486e-01     False
 A_clean vs E_clean V

## TABLE IV — the sanitisation blind spot, with uncertainty

Reviewer 3 asked for the uncertainty behind the headline comparison. Three
things are reported here: Wilson intervals on each rate, a conditional-MLE odds
ratio with its exact interval, and the same computation repeated across all ten
conditions so the reader can see whether the asymmetry is a property of one
prompt or of the model.

**The mapping below must be checked against the one used for the published
TABLE IV.** It was reconstructed from the aggregate counts, and the verification
cell that follows will say whether it reproduces the published figures.

In [12]:
MECHANISM = {
    # --- type-level guarantees -------------------------------------------
    **{k: 'Type coercion' for k in [
        'CAST-cast_float','CAST-cast_float_sort_of','CAST-cast_int',
        'CAST-cast_int_sort_of','CAST-cast_int_sort_of2',
        'CAST-func_settype_float','CAST-func_settype_int',
        'func_floatval','func_intval',
        'func_FILTER-CLEANING-number_float_filter',
        'func_FILTER-CLEANING-number_int_filter',
        'func_FILTER-VALIDATION-number_float_filter',
        'func_FILTER-VALIDATION-number_int_filter']},
    **{k: 'Whitelist' for k in [
        'ternary_white_list','whitelist_using_array','whitelist_using_array_from']},
    # --- escaping and filtering ------------------------------------------
    **{k: 'Regex validation' for k in [
        'func_preg_match-letters_numbers','func_preg_match-only_letters',
        'func_preg_match-only_numbers']},
    **{k: 'Regex replacement' for k in ['func_preg_replace','func_preg_replace2']},
    **{k: 'HTML escaping' for k in ['func_htmlentities','func_htmlspecialchars']},
    **{k: 'Quote escaping' for k in [
        'func_addslashes','func_escapeshellarg','func_mysql_real_escape_string',
        'object-func_mysql_real_escape_string',
        'object-func_mysql_real_escape_stringGetter',
        'func_FILTER-CLEANING-magic_quotes_filter']},
    'no_sanitizing': 'None',
}

MECHANISM.update({
    'func_FILTER-VALIDATION-email_filter':            'Email filter',
    'func_FILTER-CLEANING-email_filter':              'Email filter',
    'func_FILTER-CLEANING-special_chars_filter':      'Special-char filter',
    'func_FILTER-CLEANING-full_special_chars_filter': 'Special-char filter',
    'func_preg_match-no_filtering':                   'Regex validation',
})

f55['mechanism'] = f55.sanitizer.map(MECHANISM)
unmapped = f55[f55.mechanism.isna()].sanitizer.unique()
assert len(unmapped) == 0, f'unmapped sanitisers: {unmapped}'

TYPE_LEVEL = ['Type coercion', 'Whitelist']
ESCAPING   = ['Regex replacement', 'HTML escaping', 'Quote escaping']
print('safe-sample counts by mechanism:')
print(f55[(f55.true_label=='Safe') & (f55.condition=='A_clean')]
      .mechanism.value_counts().to_string())

safe-sample counts by mechanism:
mechanism
Type coercion        86
Quote escaping       25
Whitelist            19
Regex validation     19
Regex replacement    16
HTML escaping        13
None                  2


In [13]:
# --- does this mapping reproduce the published TABLE IV? ---
pub = pd.read_csv(WORK / 'TABLE4_gpt55_by_sanitizer_mechanism.csv')
pub_safe = pub[pub.TrueLabel=='Safe'].set_index('Mechanism')

chk = (f55[(f55.true_label=='Safe') & (f55.condition.isin(['A_clean','E_clean']))]
       .pivot_table(index='mechanism', columns='condition',
                    values='correct', aggfunc=['sum','size']))
rep = pd.DataFrame({
    'n_new':  chk[('size','A_clean')].astype(int),
    'n_pub':  pub_safe.n,
    'A_new':  chk[('sum','A_clean')].astype(int),
    'A_pub':  pub_safe.A_correct,
    'E_new':  chk[('sum','E_clean')].astype(int),
    'E_pub':  pub_safe.E_correct,
})
rep['match'] = (rep.n_new==rep.n_pub) & (rep.A_new==rep.A_pub) & (rep.E_new==rep.E_pub)
print(rep.to_string())

if rep.match.all():
    print('\nMapping reproduces the published TABLE IV exactly.')
else:
    print('\n*** MAPPING DIFFERS from the published table — reconcile before '
          'using any number below in the manuscript. ***')

                   n_new  n_pub  A_new  A_pub  E_new  E_pub  match
HTML escaping         13   13.0      3    3.0      3    3.0   True
None                   2    NaN      2    NaN      2    NaN  False
Quote escaping        25   23.0      3    3.0      5    5.0  False
Regex replacement     16   16.0      5    5.0      9    9.0   True
Regex validation      19   19.0     14   14.0     16   16.0   True
Type coercion         86   86.0     81   81.0     80   80.0   True
Whitelist             19   19.0     18   18.0     18   18.0   True

*** MAPPING DIFFERS from the published table — reconcile before using any number below in the manuscript. ***


In [14]:
sa = f55[(f55.true_label=='Safe') & (f55.condition=='A_clean')]
q = sa[sa.mechanism=='Quote escaping']
print(q.groupby('sanitizer').agg(n=('correct','size'), correct=('correct','sum')).to_string())

                                            n  correct
sanitizer                                             
func_FILTER-CLEANING-magic_quotes_filter    7        0
func_addslashes                             6        0
func_escapeshellarg                         2        0
func_mysql_real_escape_string               6        2
object-func_mysql_real_escape_string        2        0
object-func_mysql_real_escape_stringGetter  2        1


In [15]:
# --- blind spot across all ten conditions, with intervals ---
safe = f55[f55.true_label == 'Safe']
rows = []
for cond in CONDS:
    s = safe[safe.condition == cond]
    tl = s[s.mechanism.isin(TYPE_LEVEL)]
    es = s[s.mechanism.isin(ESCAPING)]
    a, na = int(tl.correct.sum()), len(tl)
    b, nb = int(es.correct.sum()), len(es)
    al, ah = proportion_confint(a, na, method='wilson')
    bl, bh = proportion_confint(b, nb, method='wilson')
    res = stats.contingency.odds_ratio([[a, na-a], [b, nb-b]])
    ci  = res.confidence_interval(0.95)
    rows.append({
        'Condition': cond,
        'TypeLevel': f'{a}/{na}', 'TL_rate': round(a/na,3),
        'TL_lo': round(al,3), 'TL_hi': round(ah,3),
        'Escaping': f'{b}/{nb}', 'Esc_rate': round(b/nb,3),
        'Esc_lo': round(bl,3), 'Esc_hi': round(bh,3),
        'OR': round(res.statistic,1),
        'OR_lo': round(ci.low,1), 'OR_hi': round(ci.high,1),
        'p_fisher': stats.fisher_exact([[a,na-a],[b,nb-b]])[1],
    })
t4 = pd.DataFrame(rows)
print(t4.to_string(index=False))
t4.to_csv(OUT / 'TABLE4_blindspot_all_conditions.csv', index=False)

Condition TypeLevel  TL_rate  TL_lo  TL_hi Escaping  Esc_rate  Esc_lo  Esc_hi   OR  OR_lo  OR_hi     p_fisher
  A_clean    99/105    0.943  0.881  0.974    11/54     0.204   0.118   0.329 61.3   20.5  218.8 5.127291e-22
 A_hinted   105/105    1.000  0.965  1.000    37/54     0.685   0.553   0.793  inf   10.8    inf 1.534131e-09
  B_clean    99/105    0.943  0.881  0.974    10/54     0.185   0.104   0.308 68.7   22.6  250.0 5.816693e-23
 B_hinted   105/105    1.000  0.965  1.000    37/54     0.685   0.553   0.793  inf   10.8    inf 1.534131e-09
  C_clean    99/105    0.943  0.881  0.974    16/54     0.296   0.191   0.428 37.7   13.2  127.6 9.183048e-18
 C_hinted   105/105    1.000  0.965  1.000    29/54     0.537   0.406   0.663  inf   20.6    inf 1.765785e-14
  D_clean    99/105    0.943  0.881  0.974    15/54     0.278   0.176   0.409 41.2   14.3  140.4 1.471047e-18
 D_hinted   105/105    1.000  0.965  1.000    40/54     0.741   0.611   0.839  inf    8.1    inf 7.727174e-08
  E_clean 

## TABLE V — capability spectrum

GPT-5.5 now spans ten conditions rather than two, so its observed range widens.

In [16]:
t5_55 = pd.DataFrame([{
    'Model': GPT55, 'n_conditions': 10,
    'Spec_min': t1_55.Specificity.min(), 'Spec_max': t1_55.Specificity.max(),
    'BalAcc_min': t1_55.BalAcc.min(),    'BalAcc_max': t1_55.BalAcc.max(),
    'MCC_min': t1_55.MCC.min(),          'MCC_max': t1_55.MCC.max(),
}])
legacy5 = pd.read_csv(WORK / 'TABLE5_capability_spectrum.csv')
t5 = pd.concat([legacy5[legacy5.Model != GPT55], t5_55], ignore_index=True)
print(t5.to_string(index=False))
t5.to_csv(OUT / 'TABLE5_capability_spectrum.csv', index=False)

             Model  n_conditions  Spec_min  Spec_max  BalAcc_min  BalAcc_max  MCC_min  MCC_max
GPT-3.5-turbo-0125            10       0.0     0.067       0.483       0.519      NaN      NaN
       GPT-4o-mini            10       0.0     0.022       0.500       0.511      NaN      NaN
           GPT-5.5            10       0.7     0.922       0.681       0.800    0.412    0.612


## Run-to-run variability — is the cross-session replicate consistent?

The conditions were executed in three separate sessions. The drift check gave
91.7% verdict agreement against the original `A_clean` run. That figure is only
interpretable next to the within-session agreement already measured in the
repeated-runs experiment: if they are comparable, the disagreement is the
model's own nondeterminism rather than drift between sessions.

In [17]:
stab = pd.read_csv(WORK / 'stability_merged_3runs.csv')
print('columns:', list(stab.columns))
print(stab.groupby(['model','condition']).run_id.nunique().head(10).to_string())

# pairwise verdict agreement between runs, within session
ag = []
for (m, c), g in stab.groupby(['model', 'condition']):
    p = g.pivot_table(index='sample_id', columns='run_id',
                      values='prediction', aggfunc='first')
    runs = list(p.columns)
    for i, r1 in enumerate(runs):
        for r2 in runs[i+1:]:
            both = p[[r1, r2]].dropna()
            ag.append({'model': m, 'condition': c, 'pair': f'{r1}v{r2}',
                       'n': len(both),
                       'agreement': round((both[r1]==both[r2]).mean(), 3)})
agdf = pd.DataFrame(ag)
print('\nwithin-session agreement:')
print(agdf.groupby('model').agreement.describe()[['count','mean','min','max']].to_string())
print('\ncross-session drift check: 0.917 (44/48, A_clean)')

columns: ['sample_id', 'rel_path', 'cwe', 'sanitizer', 'true_label', 'condition', 'variant', 'hint', 'run_id', 'model', 'prediction', 'cwe_type', 'confidence', 'error']
model               condition
gpt-5.5-2026-04-23  A_clean      3
                    E_clean      3

within-session agreement:
                    count      mean    min    max
model                                            
gpt-5.5-2026-04-23    6.0  0.930833  0.896  0.958

cross-session drift check: 0.917 (44/48, A_clean)


## Summary of what changed

In [18]:
print('files written to', OUT)
for f in sorted(OUT.glob('*.csv')):
    print(' ', f.name)

print('\n--- headline numbers for the manuscript ---')
best = t1_55.loc[t1_55.MCC.idxmax()]
print(f'best condition      : {best.Condition}  MCC {best.MCC}  BalAcc {best.BalAcc}')
print(f'MCC range           : {t1_55.MCC.min()} – {t1_55.MCC.max()}')
print(f'significant pairs   : {int(t2_55.sig_holm.sum())} of 45 (GPT-5.5)')
print(f'legacy significant  : {int(legacy2.sig_holm.sum())} of 90')
a = t4[t4.Condition=='A_clean'].iloc[0]
print(f'blind spot (A_clean): type-level {a.TL_rate:.3f} [{a.TL_lo}, {a.TL_hi}] vs '
      f'escaping {a.Esc_rate:.3f} [{a.Esc_lo}, {a.Esc_hi}]')
print(f'                      OR {a.OR} [{a.OR_lo}, {a.OR_hi}], p = {a.p_fisher:.2e}')

files written to /content/drive/MyDrive/LLM_Security_Paper/revision_2026/tables_r3
  TABLE1_gpt55_10conditions.csv
  TABLE2_all_models_pairwise.csv
  TABLE2_gpt55_pairwise_mcnemar.csv
  TABLE3_component_attribution.csv
  TABLE4_blindspot_all_conditions.csv
  TABLE5_capability_spectrum.csv

--- headline numbers for the manuscript ---
best condition      : A_clean  MCC 0.612  BalAcc 0.8
MCC range           : 0.412 – 0.612
significant pairs   : 6 of 45 (GPT-5.5)
legacy significant  : 0 of 90
blind spot (A_clean): type-level 0.943 [0.881, 0.974] vs escaping 0.204 [0.118, 0.329]
                      OR 61.3 [20.5, 218.8], p = 5.13e-22
